In [0]:
# ==========================================================
# DELTA LAKE MERGE IMPLEMENTATION
# STEP 1: CREATE CUSTOMER MASTER DATASET
#
# Dataset contains:
# 1. Duplicate record
# 2. Null value
# ==========================================================

customer_master = [

    (101,"Rahul Sharma","Delhi",25,"Silver"),
    (102,"Priya Gupta","Mumbai",28,"Gold"),
    (103,"Amit Verma","Pune",30,"Silver"),
    (104,"Sneha Singh","Bangalore",26,"Platinum"),

    # duplicate record
    (104,"Sneha Singh","Bangalore",26,"Platinum"),

    # null value
    (105,None,"Hyderabad",29,"Gold"),

    (106,"Rohan Mehta","Jaipur",24,"Silver"),
    (107,"Anjali Patel","Ahmedabad",31,"Gold")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "age",
    "membership"
]

master_df = spark.createDataFrame(
    customer_master,
    columns
)

print("MASTER DATASET")
display(master_df)

MASTER DATASET


customer_id,customer_name,city,age,membership
101,Rahul Sharma,Delhi,25,Silver
102,Priya Gupta,Mumbai,28,Gold
103,Amit Verma,Pune,30,Silver
104,Sneha Singh,Bangalore,26,Platinum
104,Sneha Singh,Bangalore,26,Platinum
105,null,Hyderabad,29,Gold
106,Rohan Mehta,Jaipur,24,Silver
107,Anjali Patel,Ahmedabad,31,Gold


In [0]:
# ==========================================================
# STEP 2: DATA CLEANING
#
# Remove:
# 1. Null values
# 2. Duplicate records
# ==========================================================

clean_df = (
    master_df
    .dropna()
    .dropDuplicates()
)

print("CLEANED DATASET")
display(clean_df)

CLEANED DATASET


customer_id,customer_name,city,age,membership
101,Rahul Sharma,Delhi,25,Silver
102,Priya Gupta,Mumbai,28,Gold
103,Amit Verma,Pune,30,Silver
104,Sneha Singh,Bangalore,26,Platinum
106,Rohan Mehta,Jaipur,24,Silver
107,Anjali Patel,Ahmedabad,31,Gold


In [0]:
# ==========================================================
# STEP 3: CREATE DELTA TABLE
# ==========================================================

clean_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("customer_delta")

In [0]:
# ==========================================================
# STEP 4: CREATE INCREMENTAL DATA
#
# Existing customers:
#   update
#
# New customers:
#   insert
# ==========================================================

incremental_data = [

    # existing customer update
    (102,"Priya Gupta","Chennai",29,"Platinum"),

    # existing customer update
    (103,"Amit Verma","Pune",31,"Gold"),

    # new customer
    (108,"Karan Kapoor","Delhi",27,"Silver"),

    # new customer
    (109,"Pooja Sharma","Mumbai",23,"Gold")
]

incremental_df = spark.createDataFrame(
    incremental_data,
    columns
)

display(incremental_df)

customer_id,customer_name,city,age,membership
102,Priya Gupta,Chennai,29,Platinum
103,Amit Verma,Pune,31,Gold
108,Karan Kapoor,Delhi,27,Silver
109,Pooja Sharma,Mumbai,23,Gold


In [0]:
# ==========================================================
# STEP 4: CREATE INCREMENTAL DATA
#
# Existing customers:
#   update
#
# New customers:
#   insert
# ==========================================================

incremental_data = [

    # existing customer update
    (102,"Priya Gupta","Chennai",29,"Platinum"),

    # existing customer update
    (103,"Amit Verma","Pune",31,"Gold"),

    # new customer
    (108,"Karan Kapoor","Delhi",27,"Silver"),

    # new customer
    (109,"Pooja Sharma","Mumbai",23,"Gold")
]

incremental_df = spark.createDataFrame(
    incremental_data,
    columns
)

display(incremental_df)

customer_id,customer_name,city,age,membership
102,Priya Gupta,Chennai,29,Platinum
103,Amit Verma,Pune,31,Gold
108,Karan Kapoor,Delhi,27,Silver
109,Pooja Sharma,Mumbai,23,Gold


In [0]:
# ==========================================================
# STEP 5: IMPORT DELTA TABLE
# ==========================================================

from delta.tables import *

In [0]:
# ==========================================================
# LOAD DELTA TABLE
# ==========================================================

delta_table = DeltaTable.forName(
    spark,
    "customer_delta"
)

In [0]:
# ==========================================================
# STEP 6: DELTA MERGE OPERATION
#
# Update existing customers
# Insert new customers
# ==========================================================

(
    delta_table.alias("old")
    .merge(
        incremental_df.alias("new"),
        "old.customer_id=new.customer_id"
    )

    .whenMatchedUpdate(
        set={
            "customer_name":"new.customer_name",
            "city":"new.city",
            "age":"new.age",
            "membership":"new.membership"
        }
    )

    .whenNotMatchedInsert(
        values={
            "customer_id":"new.customer_id",
            "customer_name":"new.customer_name",
            "city":"new.city",
            "age":"new.age",
            "membership":"new.membership"
        }
    )

    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# ==========================================================
# STEP 7: DISPLAY FINAL DELTA TABLE
# ==========================================================

final_df = spark.table(
    "customer_delta"
)

display(final_df)

customer_id,customer_name,city,age,membership
101,Rahul Sharma,Delhi,25,Silver
104,Sneha Singh,Bangalore,26,Platinum
106,Rohan Mehta,Jaipur,24,Silver
107,Anjali Patel,Ahmedabad,31,Gold
102,Priya Gupta,Chennai,29,Platinum
103,Amit Verma,Pune,31,Gold
108,Karan Kapoor,Delhi,27,Silver
109,Pooja Sharma,Mumbai,23,Gold


In [0]:
# ==========================================================
# VALIDATION 1
# CHECK TOTAL ROWS
# ==========================================================

print(
    "Total Rows:",
    final_df.count()
)

Total Rows: 8


In [0]:
# ==========================================================
# VALIDATION 2
# CHECK DUPLICATES
# ==========================================================

duplicates = (

    final_df
    .groupBy("customer_id")
    .count()
    .filter("count > 1")
)

display(duplicates)

customer_id,count


In [0]:
# ==========================================================
# VALIDATION 3
# CHECK NULL VALUES
# ==========================================================

from pyspark.sql.functions import *

null_values = final_df.filter(
    col("customer_name").isNull()
)

display(null_values)

customer_id,customer_name,city,age,membership


In [0]:
# ==========================================================
# CREATE MASTER CSV
# ==========================================================

master_df.toPandas().to_csv(
    "customer_master.csv",
    index=False
)

# ==========================================================
# CREATE INCREMENTAL CSV
# ==========================================================

incremental_df.toPandas().to_csv(
    "customer_incremental.csv",
    index=False
)

print("CSV files created successfully")

CSV files created successfully
